In [32]:

import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from pgmpy.estimators import HillClimbSearch, BICGauss, PC
from pgmpy.models import LinearGaussianBayesianNetwork

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)


DEVICE: cpu


## 1. User configuration

In [ ]:

# =========================
# User configuration
# =========================

DATA_PATH = Path('final_data.csv') 
TARGET_COLUMNS = ['Age', 'log_BMI', 'BP_mean', 'HR_mean', 'log_SDNN']

TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.20   # validation fraction taken from the post-test training portion

# Structure learning
MAX_INDEGREE = None            # set an integer to constrain the search
PC_VARIANT = 'stable'
PC_SIGNIFICANCE_LEVEL = 0.05

# Ridge
RIDGE_USE_CV = True
RIDGE_ALPHA_GRID = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
RIDGE_CV_FOLDS = 3

# Neural model
NN_HIDDEN_DIMS = [32, 32, 32]
NN_ACTIVATION = nn.ReLU
NN_EPOCHS = 500
NN_LR = 1e-3
NN_PATIENCE = 20
MIN_STD = 1e-3


## 2. Load data and keep selected columns

In [34]:

df = pd.read_csv(DATA_PATH)
print('Original shape:', df.shape)

if TARGET_COLUMNS:
    missing_cols = [c for c in TARGET_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f'Missing columns in dataset: {missing_cols}')
    model_vars = TARGET_COLUMNS.copy()
else:
    model_vars = df.select_dtypes(include=[np.number]).columns.tolist()

if len(model_vars) < 2:
    raise ValueError('Need at least 2 numeric variables to learn a network.')

work_df = df[model_vars].copy()
work_df = work_df.dropna().reset_index(drop=True)

print('Variables used:', model_vars)
print('Complete-case shape:', work_df.shape)
work_df.head()


Original shape: (920, 8)
Variables used: ['Age', 'log_BMI', 'BP_mean', 'HR_mean', 'log_SDNN']
Complete-case shape: (920, 5)


,Age,log_BMI,BP_mean,HR_mean,log_SDNN
0,35.0,3.135494,115.711222,70.050083,4.049750
1,35.0,3.178054,92.187697,65.779064,4.242539
2,85.0,3.332205,95.211534,92.792893,1.653724
3,55.0,3.178054,70.455644,62.763632,3.853201
4,35.0,2.995732,95.577240,65.140860,4.196542


## 3. Train / validation / test split and scaling

In [35]:

train_full_df, test_df = train_test_split(work_df, test_size=TEST_SIZE, random_state=SEED)
train_df, val_df = train_test_split(train_full_df, test_size=VAL_SIZE_WITHIN_TRAIN, random_state=SEED)

scaler = StandardScaler()
train_scaled = pd.DataFrame(scaler.fit_transform(train_df), columns=model_vars, index=train_df.index)
val_scaled = pd.DataFrame(scaler.transform(val_df), columns=model_vars, index=val_df.index)
test_scaled = pd.DataFrame(scaler.transform(test_df), columns=model_vars, index=test_df.index)

print('train:', train_scaled.shape)
print('val:', val_scaled.shape)
print('test:', test_scaled.shape)


train: (588, 5)
val: (148, 5)
test: (184, 5)


## 4. Structure learning helpers

In [36]:

def structure_to_parents(nodes, edges):
    parents = {node: [] for node in nodes}
    for u, v in edges:
        parents[v].append(u)
    return parents


def make_dag_from_edges(nodes, edges):
    dag = nx.DiGraph()
    dag.add_nodes_from(nodes)
    dag.add_edges_from(edges)
    if not nx.is_directed_acyclic_graph(dag):
        raise ValueError('Learned graph is not a DAG.')
    return dag


def sort_edges(edges):
    return sorted([tuple(e) for e in edges], key=lambda x: (str(x[0]), str(x[1])))


def learn_hc_structure(train_data, max_indegree=None):
    start = time.time()
    hc = HillClimbSearch(train_data)
    score = BICGauss(train_data)
    try:
        est = hc.estimate(scoring_method=score, max_indegree=max_indegree, show_progress=False)
    except TypeError:
        # fallback for pgmpy versions with a slightly different signature
        est = hc.estimate(scoring_method=score, max_indegree=max_indegree)
    elapsed = time.time() - start
    edges = sort_edges(est.edges())
    dag = make_dag_from_edges(train_data.columns.tolist(), edges)
    return dag, elapsed


def learn_pc_structure(train_data, variant='stable', significance_level=0.05):
    start = time.time()
    pc = PC(data=train_data)
    est = pc.estimate(
        variant=variant,
        ci_test='pearsonr',
        return_type='dag',
        significance_level=significance_level,
        show_progress=False,
    )
    elapsed = time.time() - start
    edges = sort_edges(est.edges())
    dag = make_dag_from_edges(train_data.columns.tolist(), edges)
    return dag, elapsed


## 5. Learn the two network structures from the scaled training data

In [37]:

hc_dag, hc_structure_time = learn_hc_structure(train_scaled, max_indegree=MAX_INDEGREE)
pc_dag, pc_structure_time = learn_pc_structure(train_scaled, variant=PC_VARIANT, significance_level=PC_SIGNIFICANCE_LEVEL)

structure_summaries = pd.DataFrame([
    {
        'structure_name': 'HillClimb',
        'n_nodes': hc_dag.number_of_nodes(),
        'n_edges': hc_dag.number_of_edges(),
        'structure_learning_time_sec': hc_structure_time,
        'edges': sort_edges(hc_dag.edges()),
    },
    {
        'structure_name': 'PC',
        'n_nodes': pc_dag.number_of_nodes(),
        'n_edges': pc_dag.number_of_edges(),
        'structure_learning_time_sec': pc_structure_time,
        'edges': sort_edges(pc_dag.edges()),
    },
])

structure_summaries


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Age': 'N', 'log_BMI': 'N', 'BP_mean': 'N', 'HR_mean': 'N', 'log_SDNN': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Age': 'N', 'log_BMI': 'N', 'BP_mean': 'N', 'HR_mean': 'N', 'log_SDNN': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Age': 'N', 'log_BMI': 'N', 'BP_mean': 'N', 'HR_mean': 'N', 'log_SDNN': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Age': 'N', 'log_BMI': 'N', 'BP_mean': 'N', 'HR_mean': 'N', 'log_SDNN': 'N'}


,structure_name,n_nodes,n_edges,structure_learning_time_sec,edges
0,HillClimb,5,4,0.192184,"[(Age, log_BMI), (Age, log_SDNN), (HR_mean, BP..."
1,PC,5,3,0.021342,"[(Age, log_SDNN), (BP_mean, HR_mean), (log_BMI..."


## 6. Parameter-learning helpers

In [38]:

def extract_gbn_parameters(gbn, nodes, min_std=1e-3):
    parents_dict = {}
    mu = []
    weights = []
    sigma = []

    for node in nodes:
        parent_names = list(gbn.get_parents(node))
        parents_dict[node] = parent_names

        cpd = gbn.get_cpds(node)
        mu.append(float(cpd.beta[0]))

        if len(cpd.beta) > 1:
            w = np.array(cpd.beta[1:].flatten(), dtype=float)
        else:
            w = np.array([], dtype=float)
        weights.append(w)
        sigma.append(max(float(cpd.std), min_std))

    return parents_dict, mu, weights, sigma


def fit_mle_gbn(train_data, dag):
    start = time.time()

    gbn = LinearGaussianBayesianNetwork()
    gbn.add_nodes_from(dag.nodes())
    gbn.add_edges_from(dag.edges())

    gbn.fit(train_data)
    elapsed = time.time() - start

    nodes = list(train_data.columns)
    parents_dict, mu, weights, sigma = extract_gbn_parameters(gbn, nodes, min_std=MIN_STD)

    return {
        'method': 'MLE',
        'train_time_sec': elapsed,
        'model': gbn,
        'nodes': nodes,
        'parents_dict': parents_dict,
        'mu': mu,
        'weights': weights,
        'sigma': sigma,
    }


def fit_ridge_for_node(X, y, use_cv=True, alpha_grid=None, cv_folds=3):
    if X.shape[1] == 0:
        mu = float(np.mean(y))
        resid = y - mu
        sigma = max(float(np.std(resid, ddof=1)), MIN_STD)
        return mu, np.array([], dtype=float), sigma, np.nan

    if use_cv:
        ridge = Ridge(fit_intercept=True)
        cv = KFold(n_splits=cv_folds, shuffle=True, random_state=SEED)
        grid = GridSearchCV(
            estimator=ridge,
            param_grid=alpha_grid,
            scoring='neg_mean_squared_error',
            cv=cv,
            n_jobs=-1,
        )
        grid.fit(X, y)
        model = grid.best_estimator_
        alpha_used = float(grid.best_params_['alpha'])
    else:
        model = Ridge(alpha=1.0, fit_intercept=True)
        model.fit(X, y)
        alpha_used = float(model.alpha)

    y_hat = model.predict(X)
    resid = y - y_hat
    sigma = max(float(np.std(resid, ddof=1)), MIN_STD)

    return float(model.intercept_), np.array(model.coef_, dtype=float), sigma, alpha_used


def fit_ridge_model(train_data, dag, use_cv=True, alpha_grid=None, cv_folds=3):
    start = time.time()
    nodes = list(train_data.columns)
    parents_dict = structure_to_parents(nodes, sort_edges(dag.edges()))

    mu = []
    weights = []
    sigma = []
    alpha_used = []

    for node in nodes:
        parent_names = parents_dict[node]
        y = train_data[node].values.astype(float)
        X = train_data[parent_names].values.astype(float) if parent_names else np.zeros((len(y), 0))

        node_mu, node_w, node_sigma, alpha = fit_ridge_for_node(
            X=X,
            y=y,
            use_cv=use_cv,
            alpha_grid=alpha_grid,
            cv_folds=cv_folds,
        )
        mu.append(node_mu)
        weights.append(node_w)
        sigma.append(node_sigma)
        alpha_used.append(alpha)

    elapsed = time.time() - start
    return {
        'method': 'Ridge',
        'train_time_sec': elapsed,
        'model': None,
        'nodes': nodes,
        'parents_dict': parents_dict,
        'mu': mu,
        'weights': weights,
        'sigma': sigma,
        'alpha_used': alpha_used,
    }


## 7. Neural Gaussian model

In [39]:

class NodeNN(nn.Module):
    def __init__(self, n_parents, hidden_dims, activation):
        super().__init__()
        input_dim = n_parents if n_parents > 0 else 1
        dims = [input_dim] + list(hidden_dims)

        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if activation is not None:
                layers.append(activation())

        self.feature_extractor = nn.Sequential(*layers) if layers else nn.Identity()
        final_dim = dims[-1] if len(dims) > 0 else input_dim
        self.mean_out = nn.Linear(final_dim, 1)
        self.logvar_out = nn.Linear(final_dim, 1)

    def forward(self, x):
        h = self.feature_extractor(x)
        mu = self.mean_out(h).squeeze(-1)
        log_var = self.logvar_out(h).squeeze(-1)
        return mu, log_var


def gaussian_nll(mu, log_var, target):
    log_var = torch.clamp(log_var, min=-10.0, max=10.0)
    var = torch.exp(log_var)
    constant = torch.log(torch.tensor(2.0 * np.pi, device=mu.device, dtype=mu.dtype))
    return 0.5 * torch.mean(constant + log_var + ((target - mu) ** 2) / var)


def train_nodewise_nns(df_train, df_val, parents_dict, nodes, hidden_dims, activation, device='cpu', n_epochs=500, lr=1e-3, patience=20):
    nn_models = []
    optimizers = []
    node_train_times = []

    for node in nodes:
        model = NodeNN(
            n_parents=len(parents_dict[node]),
            hidden_dims=hidden_dims,
            activation=activation,
        ).to(device)
        nn_models.append(model)
        optimizers.append(optim.Adam(model.parameters(), lr=lr))
        node_train_times.append(0.0)

    best_val_losses = [float('inf')] * len(nodes)
    patience_counters = [0] * len(nodes)
    best_states = [None] * len(nodes)
    stopped = [False] * len(nodes)

    for epoch in range(n_epochs):
        active_nodes = 0

        for i, node in enumerate(nodes):
            if stopped[i]:
                continue

            start = time.time()
            model = nn_models[i]
            optimizer = optimizers[i]
            model.train()

            x_train = torch.tensor(df_train[parents_dict[node]].values, dtype=torch.float32, device=device) if parents_dict[node] else torch.zeros((len(df_train), 1), dtype=torch.float32, device=device)
            y_train = torch.tensor(df_train[node].values, dtype=torch.float32, device=device)

            optimizer.zero_grad()
            mu_pred, log_var_pred = model(x_train)
            loss = gaussian_nll(mu_pred, log_var_pred, y_train)
            loss.backward()
            optimizer.step()

            node_train_times[i] += time.time() - start
            active_nodes += 1

            model.eval()
            with torch.no_grad():
                x_val = torch.tensor(df_val[parents_dict[node]].values, dtype=torch.float32, device=device) if parents_dict[node] else torch.zeros((len(df_val), 1), dtype=torch.float32, device=device)
                y_val = torch.tensor(df_val[node].values, dtype=torch.float32, device=device)
                mu_val, log_var_val = model(x_val)
                val_loss = gaussian_nll(mu_val, log_var_val, y_val).item()

            if val_loss < best_val_losses[i]:
                best_val_losses[i] = val_loss
                patience_counters[i] = 0
                best_states[i] = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                patience_counters[i] += 1
                if patience_counters[i] >= patience:
                    stopped[i] = True
                    if best_states[i] is not None:
                        model.load_state_dict(best_states[i])

        if all(stopped):
            break

    for i, model in enumerate(nn_models):
        if best_states[i] is not None:
            model.load_state_dict(best_states[i])
        model.eval()

    return nn_models, best_val_losses, node_train_times


def fit_nn_model(train_data, val_data, dag, hidden_dims, activation, device='cpu', n_epochs=500, lr=1e-3, patience=20):
    nodes = list(train_data.columns)
    parents_dict = structure_to_parents(nodes, sort_edges(dag.edges()))

    start = time.time()
    nn_models, best_val_losses, node_train_times = train_nodewise_nns(
        df_train=train_data,
        df_val=val_data,
        parents_dict=parents_dict,
        nodes=nodes,
        hidden_dims=hidden_dims,
        activation=activation,
        device=device,
        n_epochs=n_epochs,
        lr=lr,
        patience=patience,
    )
    elapsed = time.time() - start

    return {
        'method': 'NN',
        'train_time_sec': elapsed,
        'node_train_times': node_train_times,
        'mean_node_train_time_sec': float(np.mean(node_train_times)),
        'best_val_losses': best_val_losses,
        'model': nn_models,
        'nodes': nodes,
        'parents_dict': parents_dict,
    }


## 8. Evaluation helpers

In [40]:

def linear_gaussian_predict(eval_df, nodes, parents_dict, mu, weights, sigma):
    mu_hat = pd.DataFrame(index=eval_df.index, columns=nodes, dtype=float)
    sigma_hat = pd.DataFrame(index=eval_df.index, columns=nodes, dtype=float)

    for i, node in enumerate(nodes):
        parent_names = parents_dict[node]
        if parent_names:
            X = eval_df[parent_names].values.astype(float)
            pred_mu = mu[i] + X @ np.asarray(weights[i], dtype=float)
        else:
            pred_mu = np.full(len(eval_df), mu[i], dtype=float)
        pred_sigma = np.full(len(eval_df), max(float(sigma[i]), MIN_STD), dtype=float)

        mu_hat[node] = pred_mu
        sigma_hat[node] = pred_sigma

    return mu_hat, sigma_hat


def nn_predict(eval_df, nodes, parents_dict, nn_models, device='cpu'):
    mu_hat = pd.DataFrame(index=eval_df.index, columns=nodes, dtype=float)
    sigma_hat = pd.DataFrame(index=eval_df.index, columns=nodes, dtype=float)

    for i, node in enumerate(nodes):
        model = nn_models[i]
        parent_names = parents_dict[node]
        x = torch.tensor(eval_df[parent_names].values, dtype=torch.float32, device=device) if parent_names else torch.zeros((len(eval_df), 1), dtype=torch.float32, device=device)

        with torch.no_grad():
            mu_pred, log_var_pred = model(x)
            sigma_pred = torch.exp(0.5 * torch.clamp(log_var_pred, min=-10.0, max=10.0))

        mu_hat[node] = mu_pred.detach().cpu().numpy()
        sigma_hat[node] = np.maximum(sigma_pred.detach().cpu().numpy(), MIN_STD)

    return mu_hat, sigma_hat


def gaussian_nll_np(y, mu, sigma):
    sigma = np.maximum(np.asarray(sigma, dtype=float), MIN_STD)
    return 0.5 * np.mean(np.log(2.0 * np.pi) + 2.0 * np.log(sigma) + ((y - mu) ** 2) / (sigma ** 2))


def evaluate_predictions(y_true_df, mu_hat_df, sigma_hat_df):
    node_rows = []
    all_y = []
    all_mu = []
    all_sigma = []
    all_z = []

    for node in y_true_df.columns:
        y = y_true_df[node].values.astype(float)
        mu = mu_hat_df[node].values.astype(float)
        sigma = np.maximum(sigma_hat_df[node].values.astype(float), MIN_STD)

        rmse = np.sqrt(mean_squared_error(y, mu))
        mae = mean_absolute_error(y, mu)
        r2 = r2_score(y, mu)
        nll = gaussian_nll_np(y, mu, sigma)

        lower = mu - 1.96 * sigma
        upper = mu + 1.96 * sigma
        coverage_95 = np.mean((y >= lower) & (y <= upper))

        z = (y - mu) / sigma
        z_mean = float(np.mean(z))
        z_std = float(np.std(z, ddof=1)) if len(z) > 1 else np.nan

        node_rows.append({
            'node': node,
            'rmse': rmse,
            'mae': mae,
            'r2': r2,
            'nll': nll,
            'coverage_95': coverage_95,
            'std_resid_mean': z_mean,
            'std_resid_std': z_std,
        })

        all_y.append(y)
        all_mu.append(mu)
        all_sigma.append(sigma)
        all_z.append(z)

    node_metrics = pd.DataFrame(node_rows)

    all_y = np.concatenate(all_y)
    all_mu = np.concatenate(all_mu)
    all_sigma = np.concatenate(all_sigma)
    all_z = np.concatenate(all_z)

    overall = {
        'rmse': float(np.sqrt(mean_squared_error(all_y, all_mu))),
        'mae': float(mean_absolute_error(all_y, all_mu)),
        'r2': float(r2_score(all_y, all_mu)),
        'nll': float(gaussian_nll_np(all_y, all_mu, all_sigma)),
        'coverage_95': float(np.mean((all_y >= all_mu - 1.96 * all_sigma) & (all_y <= all_mu + 1.96 * all_sigma))),
        'std_resid_mean': float(np.mean(all_z)),
        'std_resid_std': float(np.std(all_z, ddof=1)),
    }

    return overall, node_metrics


def evaluate_fitted_model(fit_result, structure_name, test_data, device='cpu'):
    if fit_result['method'] in {'MLE', 'Ridge'}:
        mu_hat, sigma_hat = linear_gaussian_predict(
            eval_df=test_data,
            nodes=fit_result['nodes'],
            parents_dict=fit_result['parents_dict'],
            mu=fit_result['mu'],
            weights=fit_result['weights'],
            sigma=fit_result['sigma'],
        )
    elif fit_result['method'] == 'NN':
        mu_hat, sigma_hat = nn_predict(
            eval_df=test_data,
            nodes=fit_result['nodes'],
            parents_dict=fit_result['parents_dict'],
            nn_models=fit_result['model'],
            device=device,
        )
    else:
        raise ValueError(f"Unknown method: {fit_result['method']}")

    overall, node_metrics = evaluate_predictions(
        y_true_df=test_data[fit_result['nodes']],
        mu_hat_df=mu_hat,
        sigma_hat_df=sigma_hat,
    )

    overall_row = {
        'structure': structure_name,
        'method': fit_result['method'],
        'train_time_sec': fit_result['train_time_sec'],
        **overall,
    }

    node_metrics = node_metrics.copy()
    node_metrics.insert(0, 'method', fit_result['method'])
    node_metrics.insert(0, 'structure', structure_name)

    return overall_row, node_metrics, mu_hat, sigma_hat


## 9. Fit MLE, Ridge, and NN on each learned structure

In [41]:

structures = {
    'HillClimb': hc_dag,
    'PC': pc_dag,
}

fit_results = {}
for structure_name, dag in structures.items():
    print(f'Fitting models for structure: {structure_name}')
    fit_results[(structure_name, 'MLE')] = fit_mle_gbn(train_scaled, dag)
    fit_results[(structure_name, 'Ridge')] = fit_ridge_model(
        train_data=train_scaled,
        dag=dag,
        use_cv=RIDGE_USE_CV,
        alpha_grid=RIDGE_ALPHA_GRID,
        cv_folds=RIDGE_CV_FOLDS,
    )
    fit_results[(structure_name, 'NN')] = fit_nn_model(
        train_data=train_scaled,
        val_data=val_scaled,
        dag=dag,
        hidden_dims=NN_HIDDEN_DIMS,
        activation=NN_ACTIVATION,
        device=DEVICE,
        n_epochs=NN_EPOCHS,
        lr=NN_LR,
        patience=NN_PATIENCE,
    )


Fitting models for structure: HillClimb
Fitting models for structure: PC


## 10. Evaluate all fitted models on the scaled test set

In [42]:

overall_rows = []
node_metric_tables = []
prediction_store = {}

for (structure_name, method_name), fit_result in fit_results.items():
    overall_row, node_metrics, mu_hat, sigma_hat = evaluate_fitted_model(
        fit_result=fit_result,
        structure_name=structure_name,
        test_data=test_scaled,
        device=DEVICE,
    )
    overall_rows.append(overall_row)
    node_metric_tables.append(node_metrics)
    prediction_store[(structure_name, method_name)] = {
        'mu_hat': mu_hat,
        'sigma_hat': sigma_hat,
    }

overall_results = pd.DataFrame(overall_rows).sort_values(['structure', 'method']).reset_index(drop=True)
node_results = pd.concat(node_metric_tables, ignore_index=True)

overall_results


,structure,method,train_time_sec,rmse,mae,r2,nll,coverage_95,std_resid_mean,std_resid_std
0,HillClimb,MLE,0.010473,0.962786,0.711765,0.099895,1.400804,0.926087,-0.002501,1.090498
1,HillClimb,NN,0.843657,0.961777,0.709545,0.101781,1.368398,0.933696,0.012181,1.006257
2,HillClimb,Ridge,29.528886,0.962806,0.712131,0.099858,1.380363,0.934783,-0.006117,1.017815
3,PC,MLE,0.006256,0.995798,0.752623,0.037111,1.418779,0.931522,0.017482,1.038269
4,PC,NN,2.357839,0.990920,0.747409,0.046522,1.400269,0.942391,0.054402,1.003290
5,PC,Ridge,0.077888,0.995928,0.752876,0.036860,1.415639,0.935870,0.014807,1.016345


## 11. Add structure-learning times to get total pipeline time

In [43]:

structure_time_map = {
    'HillClimb': hc_structure_time,
    'PC': pc_structure_time,
}

overall_results['structure_learning_time_sec'] = overall_results['structure'].map(structure_time_map)
overall_results['total_pipeline_time_sec'] = overall_results['structure_learning_time_sec'] + overall_results['train_time_sec']

overall_results = overall_results[[
    'structure', 'method',
    'structure_learning_time_sec', 'train_time_sec', 'total_pipeline_time_sec',
    'rmse', 'mae', 'r2', 'nll', 'coverage_95', 'std_resid_mean', 'std_resid_std'
]]

overall_results


,structure,method,structure_learning_time_sec,train_time_sec,total_pipeline_time_sec,rmse,mae,r2,nll,coverage_95,std_resid_mean,std_resid_std
0,HillClimb,MLE,0.192184,0.010473,0.202657,0.962786,0.711765,0.099895,1.400804,0.926087,-0.002501,1.090498
1,HillClimb,NN,0.192184,0.843657,1.035842,0.961777,0.709545,0.101781,1.368398,0.933696,0.012181,1.006257
2,HillClimb,Ridge,0.192184,29.528886,29.721071,0.962806,0.712131,0.099858,1.380363,0.934783,-0.006117,1.017815
3,PC,MLE,0.021342,0.006256,0.027598,0.995798,0.752623,0.037111,1.418779,0.931522,0.017482,1.038269
4,PC,NN,0.021342,2.357839,2.379181,0.990920,0.747409,0.046522,1.400269,0.942391,0.054402,1.003290
5,PC,Ridge,0.021342,0.077888,0.099231,0.995928,0.752876,0.036860,1.415639,0.935870,0.014807,1.016345


## 12. Per-node results

In [44]:
node_results.head(20)

,structure,method,node,rmse,mae,r2,nll,coverage_95,std_resid_mean,std_resid_std
0,HillClimb,MLE,Age,1.024728,0.749106,-0.005772,1.443890,0.880435,0.077494,1.022829
1,HillClimb,MLE,log_BMI,0.833520,0.659142,0.100153,1.237992,0.961957,-0.170251,0.953744
2,HillClimb,MLE,BP_mean,1.010027,0.690338,0.019395,1.429486,0.940217,-0.003134,1.026782
3,HillClimb,MLE,HR_mean,1.012273,0.821872,-0.005978,1.431248,0.951087,-0.077903,1.010294
4,HillClimb,MLE,log_SDNN,0.919300,0.638367,0.312859,1.461401,0.896739,0.161288,1.368839
5,HillClimb,Ridge,Age,1.024728,0.749106,-0.005772,1.443930,0.880435,0.077560,1.023700
6,HillClimb,Ridge,log_BMI,0.832997,0.658855,0.101282,1.247248,0.978261,-0.157600,0.885312
7,HillClimb,Ridge,BP_mean,1.010525,0.692397,0.018427,1.429707,0.940217,-0.004525,1.020086
8,HillClimb,Ridge,HR_mean,1.012273,0.821872,-0.005978,1.431267,0.951087,-0.077969,1.011154
9,HillClimb,Ridge,log_SDNN,0.919330,0.638425,0.312814,1.349662,0.923913,0.131952,1.119445


## 13. Optional: inspect the learned edges

In [45]:

print('HillClimb edges:')
print(sort_edges(hc_dag.edges()))
print('PC edges:')
print(sort_edges(pc_dag.edges()))


HillClimb edges:
[('Age', 'log_BMI'), ('Age', 'log_SDNN'), ('HR_mean', 'BP_mean'), ('HR_mean', 'log_SDNN')]
PC edges:
[('Age', 'log_SDNN'), ('BP_mean', 'HR_mean'), ('log_BMI', 'Age')]


## 14. Optional: save outputs

In [46]:

OUTPUT_DIR = Path('real_data_gbn_results')
OUTPUT_DIR.mkdir(exist_ok=True)

overall_results.to_csv(OUTPUT_DIR / 'overall_results.csv', index=False)
node_results.to_csv(OUTPUT_DIR / 'per_node_results.csv', index=False)
structure_summaries.to_csv(OUTPUT_DIR / 'structure_summaries.csv', index=False)

# Save learned edges in plain text / json-friendly form
with open(OUTPUT_DIR / 'learned_edges.txt', 'w', encoding='utf-8') as f:
    f.write('HillClimb')
    for e in sort_edges(hc_dag.edges()):
        f.write(f'{e}')
    f.write('PC')
    for e in sort_edges(pc_dag.edges()):
        f.write(f'{e}')

print('Saved outputs to:', OUTPUT_DIR.resolve())


Saved outputs to: C:\Users\Aashna\Documents\Masters\Final Experiment 3\real_data_gbn_results
